# Building our candidate scale from polling institutes

In this notebook we build the scale we use to place every first-round candidate from left to right, for the five presidential elections (2002, 2007, 2012, 2017, 2022).

**The idea.** Polling institutes always list the candidates in a left-to-right order in their reports. We collected these orders ourselves in `polling_axes.csv` (one row per poll, with the link to the original report). For each poll, we give the first candidate the score 0 and the last candidate the score 1, with the others evenly spaced in between. Then, for each candidate, we take the average of their scores across all the institutes of that election.

**Our three rules (decided before looking at any model result):**
1. For 2002 we only found one institute (Ipsos). We keep it and report it as a limitation.
2. Some candidates cannot be placed reliably. Jacques Cheminade is always "unplaced", because institutes put him at the end of the list by default. Any other candidate whose positions differ by more than 0.4 between institutes is also "unplaced". We remove unplaced candidates from the polls and compute the scale again without them, so they do not push the other candidates around. Their score is left empty, and their voters will follow the baseline split in our models.
3. Inside one poll, candidates are evenly spaced. This is a simplification: it does not tell us the real ideological distance between two candidates. We report it as a limitation.

**What we get at the end:** a file `candidate_scores.csv` with one row per candidate and election.

**Where the files are.** This notebook lives in `notebooks/`. It reads `data/reference/polling_axes.csv` and writes `data/reference/candidate_scores.csv`. Run it with `make scale`.

## 1. Setup

In [1]:
# We import pandas, a library that lets us work with tables (like Excel sheets) in Python.
import pandas as pd

In [2]:
# Our three rules, written as settings so they are easy to find and change.
UNPLACED_CANDIDATES = ["jacques-cheminade"]   # Rule 2: always unplaced
MAX_RANGE = 0.4                               # Rule 2: above this disagreement between institutes, the candidate is unplaced

# Names of the input file (our collected polls) and of the output file (our final scale).
INPUT_FILE = "../data/reference/polling_axes.csv"
OUTPUT_FILE = "../data/reference/candidate_scores.csv"

## 2. Load the polls we collected

In [3]:
# We read the CSV file into a table called polls.
polls = pd.read_csv(INPUT_FILE)

# We look at the first rows to check that everything loaded correctly.
polls.head()

,year,institute,fieldwork,client,source_type,source_url,order_left_to_right
0,2002,Ipsos,2002-04-15/16,Vizzavi / Le Point / France 2 / Europe 1,institute report,https://www.ipsos.com/fr-fr/letat-de-lopinion-...,arlette-laguiller;olivier-besancenot;daniel-gl...
1,2007,Ipsos,2007-04-17/18,Ipsos (Le Point / SFR),institute report,https://www.ipsos.com/fr-fr/node/312451,gerard-schivardi;arlette-laguiller;olivier-bes...
2,2007,Ifop,2007-03-19,Paris Match,institute report,https://www.ifop.com/wp-content/uploads/2018/0...,gerard-schivardi;arlette-laguiller;olivier-bes...
3,2007,CSA,2007-02-28/03-29,Le Bleu de Profession politique,press article,https://acteurspublics.fr/articles/royal-devan...,gerard-schivardi;arlette-laguiller;olivier-bes...
4,2012,Ifop,2012-04-18/20,Europe 1 / Paris Match / Public Senat,institute report,https://www.ifop.com/wp-content/uploads/2018/0...,nathalie-arthaud;philippe-poutou;jean-luc-mele...


In [4]:
# We count how many polls we have for each election.
polls.groupby("year")["institute"].count()

year
2002    1
2007    3
2012    3
2017    7
2022    8
Name: institute, dtype: int64

In [5]:
# The order of the candidates is stored as one long text, separated by ";".
# Here we look at one example to understand the format.
print(polls.loc[0, "institute"], polls.loc[0, "year"])
print(polls.loc[0, "order_left_to_right"])

Ipsos 2002
arlette-laguiller;olivier-besancenot;daniel-gluckstein;robert-hue;lionel-jospin;christiane-taubira;jean-pierre-chevenement;noel-mamere;corinne-lepage;jean-saint-josse;francois-bayrou;alain-madelin;jacques-chirac;christine-boutin;jean-marie-le-pen;bruno-megret


## 3. A function that turns polls into scores

We will need to compute the scores twice (see section 4), so we put the calculation in a function.

**How a poll becomes positions.** If a poll has `n` candidates, the candidate in place number `i` (starting from 0) gets the position `i / (n - 1)`. Example with 5 candidates: the positions are 0, 0.25, 0.5, 0.75 and 1.

**How positions become a score.** For each candidate in each election, we compute:
- `score`: the average position across institutes (this is our scale),
- `lowest` and `highest`: the lowest and highest position given by an institute,
- `range`: the difference between the two. A big range means the institutes disagree about this candidate.

The function takes a list of candidates to leave out. When a candidate is left out, the others are spread again between 0 and 1 without them.

In [6]:
def compute_scores(polls, removed):
    # removed is a list of (year, candidate_id) pairs that we do not want in the polls.

    # Step 1: one line per candidate and per poll, with their position between 0 and 1.
    positions = []
    for index, poll in polls.iterrows():
        # We split the long text into a list of candidates.
        all_candidates = poll["order_left_to_right"].split(";")

        # We keep only the candidates that are not removed for this election.
        candidates = []
        for candidate in all_candidates:
            if (poll["year"], candidate) not in removed:
                candidates.append(candidate)

        # We give each remaining candidate their position, from left (0) to right (1).
        n = len(candidates)
        for i in range(n):
            positions.append({
                "year": poll["year"],
                "institute": poll["institute"],
                "candidate_id": candidates[i],
                "position": i / (n - 1),
            })
    positions = pd.DataFrame(positions)

    # Step 2: we group by election and candidate, then compute the average, lowest and highest position.
    scores = positions.groupby(["year", "candidate_id"])["position"].agg(["mean", "min", "max", "count"])

    # We give the columns clearer names.
    scores = scores.rename(columns={"mean": "score", "min": "lowest", "max": "highest", "count": "n_institutes"})

    # groupby puts year and candidate_id aside as an "index". We turn them back into normal columns.
    scores = scores.reset_index()

    # The range is the gap between the highest and the lowest position.
    scores["range"] = scores["highest"] - scores["lowest"]
    return scores

## 4. First pass: compute with everybody, then find who we cannot place

A candidate is "unplaced" if they are in our list of always-unplaced candidates, or if the institutes disagree too much about them (range above 0.4).

In [7]:
# First pass: nobody is removed.
first_pass = compute_scores(polls, [])

# We go through every candidate and collect the ones we cannot place, with the reason.
unplaced = []
for index, row in first_pass.iterrows():
    if row["candidate_id"] in UNPLACED_CANDIDATES:
        unplaced.append({"year": row["year"], "candidate_id": row["candidate_id"],
                         "n_institutes": row["n_institutes"], "range": row["range"],
                         "reason": "listed at the end of the poll by default"})
    elif row["range"] > MAX_RANGE:
        unplaced.append({"year": row["year"], "candidate_id": row["candidate_id"],
                         "n_institutes": row["n_institutes"], "range": row["range"],
                         "reason": "institutes disagree (range above " + str(MAX_RANGE) + ")"})

unplaced = pd.DataFrame(unplaced)
unplaced

,year,candidate_id,n_institutes,range,reason
0,2012,jacques-cheminade,3,0.111111,listed at the end of the poll by default
1,2017,jacques-cheminade,7,0.200000,listed at the end of the poll by default
2,2017,jean-lassalle,7,0.500000,institutes disagree (range above 0.4)


## 5. Second pass: compute again without the unplaced candidates

An unplaced candidate should not push the others around. So we remove them from every poll of their election and spread the remaining candidates again between 0 and 1.

In [8]:
# We make the list of (year, candidate) pairs to remove.
to_remove = []
for index, row in unplaced.iterrows():
    to_remove.append((row["year"], row["candidate_id"]))

# Second pass: same calculation, without the unplaced candidates.
scores = compute_scores(polls, to_remove)
scores["status"] = "placed"
scores["reason"] = ""

# Check: after removing them, no placed candidate should have a range above our limit.
print("Placed candidates with a range above", MAX_RANGE, ":", (scores["range"] > MAX_RANGE).sum())

Placed candidates with a range above 0.4 : 0


In [9]:
# We add the unplaced candidates back at the end of the table, so the file still has every candidate.
# Their score, lowest and highest stay empty on purpose: our models must not use a number for them.
unplaced["status"] = "unplaced"
scores = pd.concat([scores, unplaced], ignore_index=True)   # concat sticks two tables together, one under the other

# Check: we still have one row per candidate and per election.
print("Rows:", len(scores), "| rows in the first pass:", len(first_pass))

Rows: 61 | rows in the first pass: 61


**How much did the second pass change?** We compare each placed candidate's score before and after removing the unplaced candidates.

In [10]:
# We put the old score next to the new one, for the same election and candidate.
# (We round only what we display. The saved file keeps every decimal: see section 7.)
compare = scores.merge(first_pass[["year", "candidate_id", "score"]], on=["year", "candidate_id"], suffixes=("", "_first_pass"))
compare["score_first_pass"] = compare["score_first_pass"].round(3)
compare["change"] = (compare["score"] - compare["score_first_pass"]).round(3)

# We show only the candidates whose score moved by more than 0.03.
compare[compare["change"].abs() > 0.03][["year", "candidate_id", "score_first_pass", "score", "change"]]

,year,candidate_id,score_first_pass,score,change
28,2012,eva-joly,0.444,0.500000,0.056
29,2012,francois-bayrou,0.556,0.625000,0.069
30,2012,francois-hollande,0.333,0.375000,0.042
32,2012,marine-le-pen,0.926,1.000000,0.074
34,2012,nicolas-dupont-aignan,0.778,0.875000,0.097
35,2012,nicolas-sarkozy,0.667,0.750000,0.083
37,2017,benoit-hamon,0.300,0.375000,0.075
38,2017,emmanuel-macron,0.400,0.500000,0.100
39,2017,francois-asselineau,0.843,0.928571,0.086
40,2017,francois-fillon,0.571,0.625000,0.054


## 6. Look at the final scale, election by election

In [11]:
# We sort the table by election, then from left (0) to right (1). Unplaced candidates (empty score) go last.
scores = scores.sort_values(["year", "score"], na_position="last").reset_index(drop=True)

# We print one small table per election.
for year in scores["year"].unique():
    print("Election", year, "-", int(scores[scores["year"] == year]["n_institutes"].max()), "institute(s)")
    print(scores[scores["year"] == year][["candidate_id", "score", "range", "status"]].round(3).to_string(index=False))
    print()

Election 2002 - 1 institute(s)
           candidate_id  score  range status
      arlette-laguiller  0.000    0.0 placed
     olivier-besancenot  0.067    0.0 placed
      daniel-gluckstein  0.133    0.0 placed
             robert-hue  0.200    0.0 placed
          lionel-jospin  0.267    0.0 placed
     christiane-taubira  0.333    0.0 placed
jean-pierre-chevenement  0.400    0.0 placed
            noel-mamere  0.467    0.0 placed
         corinne-lepage  0.533    0.0 placed
       jean-saint-josse  0.600    0.0 placed
        francois-bayrou  0.667    0.0 placed
          alain-madelin  0.733    0.0 placed
         jacques-chirac  0.800    0.0 placed
       christine-boutin  0.867    0.0 placed
      jean-marie-le-pen  0.933    0.0 placed
           bruno-megret  1.000    0.0 placed

Election 2007 - 3 institute(s)
        candidate_id  score  range status
    gerard-schivardi  0.000    0.0 placed
   arlette-laguiller  0.091    0.0 placed
  olivier-besancenot  0.182    0.0 placed
 mar

## 7. Save our scale

This file is what our models (H1, H2 and H3) will read to know where each candidate is.

We save every decimal, without rounding. Rounding can break real ties: in 2002, Christine Boutin is exactly halfway between Jacques Chirac and Jean-Marie Le Pen (13/15 between 12/15 and 14/15). Rounded to 3 decimals, she would look slightly closer to Le Pen, and H1 would send all her voters to him instead of splitting them.

In [12]:
# We save the table as a CSV file, without the row numbers.
scores.to_csv(OUTPUT_FILE, index=False)
print("Saved", len(scores), "rows to", OUTPUT_FILE)

Saved 61 rows to ../data/reference/candidate_scores.csv


## 8. Limits we need to mention in the report

- **2002 relies on one institute only (Ipsos).** The other institutes' original reports were not available online.
- **Evenly spaced positions.** Inside one poll, two neighbours are always the same distance apart. For example, in 2007 the institutes list the PS before the Greens, so Dominique Voynet ends up slightly to the right of Ségolène Royal. This comes from how institutes order their tables, not from a real ideological difference.
- **Scores are not comparable between elections.** A score of 0.5 in 2002 (16 candidates) does not mean the same thing as 0.5 in 2012 (10 candidates). This is fine for H1 and H2, which compare candidates within the same election.
- **Two sources are second-hand:** CSA 2007 comes from a press article, and 2017 and 2022 come from the tables published by Delemazure, Dong, Peters and Tydrichová (2024).